# LeetCode 2475 — Number of Unequal Triplets in Array bằng Python

Link đề: https://leetcode.com/problems/number-of-unequal-triplets-in-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting, Combinatorics

Skeleton LeetCode:

```python
class Solution:
    def unequalTriplets(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **bộ ba chỉ số** `(i, j, k)` với `i < j < k` mà ba giá trị `nums[i]`, `nums[j]`, `nums[k]` **đôi một khác nhau**.
2. Viết được cách brute force ba vòng lặp và nhận ra nó là `O(n^3)`.
3. Chuyển bài toán từ "đếm theo chỉ số" sang "đếm theo **nhóm giá trị**" bằng `Counter`: chọn ba nhóm khác nhau rồi nhân số phần tử của chúng.
4. Viết 4 cách: brute force, sắp xếp + nhóm (trái × giữa × phải), quy hoạch động theo tần suất (đa thức đối xứng bậc 3), và bù trừ (tổng trừ các bộ có trùng); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: đếm theo giá trị thay vì chỉ số, chỉ so sánh hai phần tử kề nhau, quên trừ trường hợp cả ba bằng nhau.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `nums` (chỉ số bắt đầu từ 0). Hãy đếm số **bộ ba** `(i, j, k)` thoả mãn đồng thời:

- `0 <= i < j < k < nums.length`
- `nums[i] != nums[j]`, `nums[i] != nums[k]` và `nums[j] != nums[k]` (ba số đôi một khác nhau).

Trả về số bộ ba đó.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[4,4,2,4,3]` | 3 | `(0,2,4)`, `(1,2,4)`, `(2,3,4)`: mỗi bộ gồm đúng ba giá trị `4, 2, 3` |
| 2 | `[1,1,1,1,1]` | 0 | không có ba giá trị khác nhau |

Constraints: `3 <= nums.length <= 100`; `1 <= nums[i] <= 1000`.

In [1]:
nums = [4, 4, 2, 4, 3]
n = len(nums)
found = []
for i in range(n):
    for j in range(i + 1, n):
        for k in range(j + 1, n):
            if nums[i] != nums[j] and nums[i] != nums[k] and nums[j] != nums[k]:
                found.append((i, j, k))
print("các bộ chỉ số hợp lệ:", found)
print("số lượng:", len(found))

các bộ chỉ số hợp lệ: [(0, 2, 4), (1, 2, 4), (2, 3, 4)]
số lượng: 3


## 3. Kiến thức nền

### 3.1. Ba vòng `for` lồng nhau và điều kiện `i < j < k`

Để duyệt mọi bộ ba **chỉ số** tăng dần, ta cho `j` bắt đầu từ `i + 1` và `k` bắt đầu từ `j + 1`. Cách này không lặp lại cùng một bộ theo thứ tự khác (khác với việc duyệt `range(n)` cho cả ba biến, sẽ đếm mỗi bộ 6 lần).

In [2]:
n = 4
triples = [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)]
print(triples, len(triples))

[(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)] 4


### 3.2. `collections.Counter` — đếm tần suất

`Counter(nums)` là một dict `{giá trị: số lần xuất hiện}`. `.values()` cho các số lần, `.items()` cho cặp `(giá trị, số lần)`. Đây là cách nhanh nhất để "gom các số bằng nhau vào một nhóm".

In [3]:
from collections import Counter
cnt = Counter([4, 4, 2, 4, 3])
print(cnt)
print(list(cnt.values()), len(cnt))
for value, times in cnt.items():
    print(f"giá trị {value} xuất hiện {times} lần")

Counter({4: 3, 2: 1, 3: 1})
[3, 1, 1] 3
giá trị 4 xuất hiện 3 lần
giá trị 2 xuất hiện 1 lần
giá trị 3 xuất hiện 1 lần


### 3.3. Nguyên lý nhân và tổ hợp `math.comb`

Nếu chọn một phần tử từ nhóm A (có `a` phần tử), một từ nhóm B (`b` phần tử), một từ nhóm C (`c` phần tử) thì có `a * b * c` cách (nguyên lý nhân). Còn `math.comb(n, k)` là số cách chọn `k` phần tử từ `n` phần tử **không phân biệt thứ tự**: `C(n, k) = n! / (k! (n-k)!)`. Ví dụ `C(5, 3) = 10`.

In [4]:
import math
print(math.comb(5, 3), math.comb(3, 2), math.comb(2, 3))
print(3 * 1 * 1)

10 3 0
3


### 3.4. Sắp xếp để các số bằng nhau đứng cạnh nhau

Sau `sorted(nums)`, các giá trị bằng nhau tạo thành **đoạn liên tiếp**. Khi đó với một đoạn `[l, r)` ta biết ngay có `l` phần tử **bên trái** (nhỏ hơn) và `n - r` phần tử **bên phải** (lớn hơn); cả hai bên đều khác giá trị của đoạn.

In [5]:
s = sorted([4, 4, 2, 4, 3])
print(s)
i = 0
while i < len(s):
    j = i
    while j < len(s) and s[j] == s[i]:
        j += 1
    print(f"giá trị {s[i]}: đoạn [{i}, {j}) -> {i} phần tử bên trái, {len(s) - j} bên phải")
    i = j

[2, 3, 4, 4, 4]
giá trị 2: đoạn [0, 1) -> 0 phần tử bên trái, 4 bên phải
giá trị 3: đoạn [1, 2) -> 1 phần tử bên trái, 3 bên phải
giá trị 4: đoạn [2, 5) -> 2 phần tử bên trái, 0 bên phải


### 3.5. Cập nhật tuần tự các "tổng tích" bằng ba biến

Giả sử ta duyệt từng nhóm với kích thước `c`. Ta giữ `e1` = số cách chọn 1 phần tử từ các nhóm đã xử lý, `e2` = số cách chọn 2 phần tử **từ hai nhóm khác nhau**, `e3` = số cách chọn 3 từ ba nhóm khác nhau. Khi thêm nhóm mới `c`: `e3 += e2 * c`, `e2 += e1 * c`, `e1 += c` (thứ tự cập nhật quan trọng: dùng giá trị **cũ** của `e2`, `e1`).

In [6]:
e1 = e2 = e3 = 0
for c in [3, 1, 1]:           # kích thước các nhóm: 4 xuất hiện 3 lần, 2 một lần, 3 một lần
    e3 += e2 * c
    e2 += e1 * c
    e1 += c
    print(f"sau nhóm c={c}: e1={e1}, e2={e2}, e3={e3}")

sau nhóm c=3: e1=3, e2=0, e3=0
sau nhóm c=1: e1=4, e2=3, e3=0
sau nhóm c=1: e1=5, e2=7, e3=3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Cách thẳng nhất: thử **mọi** bộ ba chỉ số và kiểm tra ba số có đôi một khác nhau không.

Quan sát cho `[4, 4, 2, 4, 3]` (5 phần tử, `C(5,3) = 10` bộ ba chỉ số):

```
giá trị:  4  4  2  4  3
chỉ số :  0  1  2  3  4
bộ ba hợp lệ phải chứa đúng một số 4, số 2 và số 3
chọn số 4 nào: 3 cách (chỉ số 0, 1 hoặc 3) ; số 2: 1 cách ; số 3: 1 cách -> 3 * 1 * 1 = 3
```

Nhận xét: bộ ba hợp lệ **luôn gồm đúng ba giá trị khác nhau**, nên đáp án chính là tổng, trên mọi cách chọn **ba giá trị phân biệt** `x, y, z`, của `cnt[x] * cnt[y] * cnt[z]`. Giá trị của chỉ số (`i < j < k`) không còn quan trọng, vì mỗi tập ba chỉ số khác nhau chỉ đếm một lần.

## 5. Cách 1 — Brute Force / baseline

Ba vòng lặp: với mỗi `i < j < k` kiểm tra ba cặp so sánh. Giữ nguyên đúng định nghĩa của đề nên rất dễ tin cậy; ta còn dùng nó làm **oracle** (đáp án chuẩn) để kiểm tra các cách khác.

Nút thắt: số bộ ba là `C(n, 3) ≈ n^3 / 6`, mỗi bộ kiểm tra `O(1)` → `O(n^3)`. Với `n <= 100` còn chạy được (`~161.700` bộ) nhưng nếu `n = 10^5` thì hoàn toàn không thể.

In [7]:
class SolutionBrute:
    def unequalTriplets(self, nums):
        n = len(nums)
        total = 0
        for i in range(n):
            for j in range(i + 1, n):
                for k in range(j + 1, n):
                    if nums[i] != nums[j] and nums[i] != nums[k] and nums[j] != nums[k]:
                        total += 1
        return total

print(SolutionBrute().unequalTriplets([4, 4, 2, 4, 3]))
print(SolutionBrute().unequalTriplets([1, 1, 1, 1, 1]))

3
0


**Độ phức tạp:** Time `O(n^3)`, Space `O(1)`. Chạy đúng nhưng chậm khi `n` lớn.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** gom các số bằng nhau thành nhóm. Một bộ ba hợp lệ = chọn **mỗi nhóm khác nhau đúng một phần tử**, ở **ba** nhóm khác nhau.

**Cách nghĩ theo nhóm ở giữa** (sau khi sắp xếp): xét nhóm có `c` phần tử, với `left` phần tử đứng trước và `right = n - left - c` phần tử đứng sau. Nếu **lấy nhóm này làm giá trị ở giữa** (giá trị trung vị về mặt thứ tự), ta chọn một phần tử ở bên trái (nhỏ hơn), một trong nhóm, một ở bên phải (lớn hơn): `left * c * right` cách. Cộng qua mọi nhóm sẽ đếm mỗi bộ ba hợp lệ **đúng một lần** (mỗi bộ ba có đúng một giá trị ở giữa).

**Quy tắc "nếu… thì…":**

- Nếu nhóm là nhỏ nhất (`left = 0`) hoặc lớn nhất (`right = 0`) → đóng góp `0` (không thể là giá trị ở giữa).
- Nếu có `k` nhóm → chỉ cần duyệt `k` nhóm một lần, mỗi nhóm `O(1)` sau khi sắp xếp/đếm.

Invariant khi duyệt nhóm theo thứ tự tăng dần: `left` = tổng kích thước các nhóm đã duyệt, luôn tăng thêm `c` sau mỗi nhóm.

In [8]:
from collections import Counter
cnt = Counter([4, 4, 2, 4, 3])
n = 5
left = 0
answer = 0
for value in sorted(cnt):
    c = cnt[value]
    right = n - left - c
    print(f"giá trị {value}: left={left}, c={c}, right={right} -> đóng góp {left * c * right}")
    answer += left * c * right
    left += c
print("đáp án:", answer)

giá trị 2: left=0, c=1, right=4 -> đóng góp 0
giá trị 3: left=1, c=1, right=3 -> đóng góp 3
giá trị 4: left=2, c=3, right=0 -> đóng góp 0
đáp án: 3


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) brute force `O(n^3)`; (2) `Counter` + nhóm ở giữa `left * c * right`; (3) quy hoạch động tuần tự `e1, e2, e3`; (4) bù trừ: tổng tất cả bộ ba trừ các bộ có hai hoặc ba số bằng nhau.

### Cách 2 — `Counter` + nhóm ở giữa

Đếm tần suất, sắp xếp các giá trị phân biệt tăng dần, rồi áp dụng công thức `left * c * right` cho mỗi nhóm như ở mục 6. Mỗi bộ ba hợp lệ được đếm đúng khi nhóm của giá trị ở giữa được xét.

In [9]:
class SolutionMiddle:
    def unequalTriplets(self, nums):
        cnt = Counter(nums)
        n = len(nums)
        left = 0
        total = 0
        for value in sorted(cnt):
            c = cnt[value]
            right = n - left - c
            total += left * c * right
            left += c
        return total

print(SolutionMiddle().unequalTriplets([4, 4, 2, 4, 3]))
print(SolutionMiddle().unequalTriplets([1, 1, 1, 1, 1]))

3
0


**Độ phức tạp Cách 2:** Time `O(n + k log k)` với `k` là số giá trị phân biệt (`k <= n`), tức `O(n log n)` trong trường hợp xấu; Space `O(k)`.

### Cách 3 — Quy hoạch động theo tần suất (`e1`, `e2`, `e3`)

Không cần sắp xếp các giá trị! Duyệt nhóm theo **thứ tự bất kỳ** và cập nhật ba biến như ở mục 3.5. `e3` cuối cùng chính là đa thức đối xứng bậc 3 của các kích thước nhóm = tổng `cnt[x] * cnt[y] * cnt[z]` trên mọi cách chọn ba nhóm phân biệt.

In [10]:
class SolutionDP:
    def unequalTriplets(self, nums):
        e1 = e2 = e3 = 0
        for c in Counter(nums).values():
            e3 += e2 * c      # chọn nhóm này làm phần tử thứ ba
            e2 += e1 * c      # ... làm phần tử thứ hai
            e1 += c           # ... làm phần tử đầu
        return e3

print(SolutionDP().unequalTriplets([4, 4, 2, 4, 3]))
print(SolutionDP().unequalTriplets([1, 1, 1, 1, 1]))

3
0


**Độ phức tạp Cách 3:** Time `O(n)` (đếm `Counter` + duyệt `k` nhóm), Space `O(k)`.

### Cách 4 — Bù trừ (trừ các bộ ba có số trùng)

Tổng số bộ ba chỉ số là `C(n, 3)`. Mỗi bộ ba **không hợp lệ** thuộc đúng một trong hai loại:

- **Có đúng hai số bằng nhau** (số còn lại khác): với mỗi giá trị có `c` phần tử, chọn 2 trong `c` rồi 1 trong `n - c` phần tử còn lại: `C(c, 2) * (n - c)`.
- **Cả ba số bằng nhau**: `C(c, 3)`.

Vậy đáp án = `C(n,3) - Σ [C(c,2)*(n-c) + C(c,3)]`.

In [11]:
import math

class SolutionComplement:
    def unequalTriplets(self, nums):
        n = len(nums)
        total = math.comb(n, 3)
        for c in Counter(nums).values():
            total -= math.comb(c, 2) * (n - c)    # đúng hai số bằng nhau
            total -= math.comb(c, 3)              # cả ba bằng nhau
        return total

print(SolutionComplement().unequalTriplets([4, 4, 2, 4, 3]))
print(SolutionComplement().unequalTriplets([1, 1, 1, 1, 1]))

3
0


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(k)`. Hay khi bạn quen đếm bù.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [4, 4, 2, 4, 3]`: nhóm theo giá trị tăng dần.

| nhóm (giá trị) | `c` | `left` | `right = n - left - c` | đóng góp `left*c*right` | tổng |
|---|---|---|---|---|---|
| 2 | 1 | 0 | 4 | 0 | 0 |
| 3 | 1 | 1 | 3 | 3 | 3 |
| 4 | 3 | 2 | 0 | 0 | 3 |

Nhóm `3` ở giữa: có `1` số nhỏ hơn (số 2), có `3` số lớn hơn (ba số 4) → 3 bộ ba. Kết quả `3`.

In [12]:
def trace_middle(nums):
    cnt = Counter(nums)
    n = len(nums)
    print("tần suất:", dict(sorted(cnt.items())))
    left = 0
    total = 0
    for value in sorted(cnt):
        c = cnt[value]
        right = n - left - c
        gain = left * c * right
        total += gain
        print(f"giá trị {value}: c={c} left={left} right={right} -> +{gain} (tổng {total})")
        left += c
    return total

print("kết quả:", trace_middle([4, 4, 2, 4, 3]))
print("kết quả:", trace_middle([1, 3, 3, 5, 5, 5, 7]))

tần suất: {2: 1, 3: 1, 4: 3}
giá trị 2: c=1 left=0 right=4 -> +0 (tổng 0)
giá trị 3: c=1 left=1 right=3 -> +3 (tổng 3)
giá trị 4: c=3 left=2 right=0 -> +0 (tổng 3)
kết quả: 3
tần suất: {1: 1, 3: 2, 5: 3, 7: 1}
giá trị 1: c=1 left=0 right=6 -> +0 (tổng 0)
giá trị 3: c=2 left=1 right=4 -> +8 (tổng 8)
giá trị 5: c=3 left=3 right=1 -> +9 (tổng 17)
giá trị 7: c=1 left=6 right=0 -> +0 (tổng 17)
kết quả: 17


## 9. Code nộp LeetCode

In [13]:
from typing import List
from collections import Counter

class Solution:
    def unequalTriplets(self, nums: List[int]) -> int:
        cnt = Counter(nums)
        n = len(nums)
        left = 0
        total = 0
        for value in sorted(cnt):
            c = cnt[value]
            right = n - left - c
            total += left * c * right
            left += c
        return total

print(Solution().unequalTriplets([4, 4, 2, 4, 3]), Solution().unequalTriplets([1, 1, 1, 1, 1]))

3 0


## 10. Vì sao thuật toán đúng

- **Mỗi bộ ba hợp lệ có ba giá trị phân biệt `x < y < z`.** Với bộ ba đó, `y` là giá trị ở giữa duy nhất. Khi xử lý nhóm `y`, ta chọn một phần tử nhóm `y`, một phần tử có giá trị nhỏ hơn (trong `left`) và một phần tử có giá trị lớn hơn (trong `right`) — bộ ba này được đếm đúng một lần, ở nhóm `y`.
- **Mỗi lần đếm tương ứng một bộ ba hợp lệ:** ba phần tử lấy từ ba vị trí khác nhau, giá trị nhỏ hơn / bằng / lớn hơn → ba giá trị khác nhau. Không bộ ba chỉ số nào bị tính hai lần vì ta đếm **tập** chỉ số (không thứ tự), mà đề `i < j < k` chỉ là cách sắp thứ tự cho tập đó.
- **Cách 3 (DP):** `e_t` sau khi xử lý một số nhóm = số cách chọn `t` phần tử từ `t` nhóm phân biệt. Khi thêm nhóm `c`: hoặc không dùng nhóm (giữ nguyên `e_t`), hoặc dùng đúng một phần tử của nó cho vị trí thứ `t` (`e_{t-1} * c`). Do cập nhật `e3` trước `e2` trước `e1` nên ta luôn dùng giá trị **trước khi** thêm nhóm, đảm bảo mỗi nhóm chỉ dùng tối đa một lần.
- **Cách 4:** phân loại bộ ba theo số cặp bằng nhau: không cặp nào (hợp lệ), đúng một cặp bằng (hai số bằng, số kia khác), hoặc cả ba bằng. Hai loại sau đếm bằng `C(c,2)*(n-c)` và `C(c,3)` (không giao nhau và phủ hết).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `k` = số giá trị phân biệt (`k <= n`).

| Cách | Time | Space |
|---|---|---|
| 1. Brute force 3 vòng lặp | `O(n^3)` | `O(1)` |
| 2. `Counter` + nhóm ở giữa | `O(n + k log k)` | `O(k)` |
| 3. DP `e1, e2, e3` | `O(n)` | `O(k)` |
| 4. Bù trừ | `O(n)` | `O(k)` |

Với `n <= 100` mọi cách đều rất nhanh; tuy vậy cách 3 là tuyến tính và không cần sắp xếp nên mở rộng được cho `n` lớn. Số nguyên Python không tràn, còn ở C++/Java đáp án tối đa `~n^3/6` nên cần `long long` khi `n` lớn.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 3` nhỏ nhất, tất cả khác nhau, tất cả bằng nhau, hai giá trị).

In [14]:
impls = [SolutionBrute(), SolutionMiddle(), SolutionDP(), SolutionComplement(), Solution()]
tests = [
    ([4, 4, 2, 4, 3], 3),
    ([1, 1, 1, 1, 1], 0),
    ([1, 2, 3], 1),                       # n = 3, ba giá trị khác nhau
    ([1, 1, 2], 0),                       # n = 3, có trùng
    ([5, 5, 5], 0),                       # n = 3, tất cả bằng nhau
    ([1, 2, 3, 4], 4),                    # tất cả khác nhau: C(4,3) = 4
    ([1, 2, 1], 0),                       # hai kề nhau khác nhau nhưng đầu = cuối (bắt lỗi chỉ so sánh kề)
    ([1, 1, 2, 2], 0),                    # chỉ hai giá trị phân biệt
    ([1, 1, 2, 2, 3], 4),                 # nhóm 3 một phần tử: 2*2*1
    ([1, 2, 3, 4, 5, 6], 20),             # C(6,3) = 20
    ([1000, 1000, 1, 1, 500], 4),         # giá trị biên 1 và 1000
    ([7, 7, 7, 7, 1, 2], 4),              # chọn một trong bốn số 7: 4 * 1 * 1
]
for nums, exp in tests:
    assert 3 <= len(nums) <= 100 and all(1 <= v <= 1000 for v in nums)
    for im in impls:
        assert im.unequalTriplets(list(nums)) == exp, (type(im).__name__, nums, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: ngẫu nhiên có seed, đúng constraints (`3 <= n <= 100`, `1 <= nums[i] <= 1000`), đối chiếu mọi cách với brute force.

In [15]:
import random
random.seed(2475)

for t in range(600):
    n = random.randint(3, 40) if t % 10 else random.randint(60, 100)
    hi_val = random.choice([2, 3, 5, 10, 1000])
    nums = [random.randint(1, hi_val) for _ in range(n)]
    assert 3 <= len(nums) <= 100 and all(1 <= v <= 1000 for v in nums)
    exp = SolutionBrute().unequalTriplets(nums)
    for im in impls[1:]:
        assert im.unequalTriplets(list(nums)) == exp, (type(im).__name__, nums)
print("600 test ngẫu nhiên OK")

600 test ngẫu nhiên OK


Test 3: tính chất — kết quả không âm; hoán vị mảng không đổi kết quả; hàm không sửa mảng; nếu mọi số khác nhau thì đáp án là `C(n, 3)`.

In [16]:
random.seed(7)
for _ in range(300):
    n = random.randint(3, 50)
    nums = [random.randint(1, 8) for _ in range(n)]
    base = Solution().unequalTriplets(nums)
    assert base >= 0
    shuffled = list(nums)
    random.shuffle(shuffled)
    assert Solution().unequalTriplets(shuffled) == base
    snapshot = list(nums)
    Solution().unequalTriplets(nums)
    assert nums == snapshot
for n in range(3, 60):
    distinct = random.sample(range(1, 1001), n)
    assert Solution().unequalTriplets(distinct) == math.comb(n, 3)
print("tính chất (không âm, bất biến theo hoán vị, không sửa input, mọi số khác nhau -> C(n,3)): OK")

tính chất (không âm, bất biến theo hoán vị, không sửa input, mọi số khác nhau -> C(n,3)): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đếm theo giá trị phân biệt thay vì theo chỉ số

In [17]:
def wrong_distinct_values(nums):
    return math.comb(len(set(nums)), 3)       # SAI: bỏ qua số lần xuất hiện

nums = [4, 4, 2, 4, 3]
print("sai :", wrong_distinct_values(nums))
print("đúng:", Solution().unequalTriplets(nums))
assert wrong_distinct_values(nums) != Solution().unequalTriplets(nums)

sai : 1
đúng: 3


Vì sao sai: đề đếm **chỉ số**. Giá trị `4` xuất hiện 3 lần nên có 3 cách chọn "một số 4" khác nhau. `C(3, 3) = 1` chỉ đếm tập giá trị, không đếm vị trí. **Sửa:** nhân với số lần xuất hiện (`cnt[x] * cnt[y] * cnt[z]`).

### Lỗi 2: chỉ so sánh các phần tử kề nhau (thiếu `nums[i] != nums[k]`)

In [18]:
def wrong_adjacent_only(nums):
    n = len(nums)
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] != nums[j] and nums[j] != nums[k]:    # SAI: thiếu nums[i] != nums[k]
                    total += 1
    return total

nums = [1, 2, 1]
print("sai :", wrong_adjacent_only(nums))
print("đúng:", Solution().unequalTriplets(nums))
assert wrong_adjacent_only(nums) != Solution().unequalTriplets(nums)

sai : 1
đúng: 0


Vì sao sai: "ba số đôi một khác nhau" cần **ba** phép so sánh `(i,j)`, `(i,k)`, `(j,k)`. Chỉ kiểm tra hai cặp kề nhau cho phép `nums[i] == nums[k]`. **Sửa:** thêm điều kiện còn thiếu.

### Lỗi 3: bù trừ nhưng quên trừ trường hợp cả ba bằng nhau

In [19]:
def wrong_forgot_triple(nums):
    n = len(nums)
    total = math.comb(n, 3)
    for c in Counter(nums).values():
        total -= math.comb(c, 2) * (n - c)       # SAI: quên trừ comb(c, 3)
    return total

nums = [1, 1, 1, 1, 1]
print("sai :", wrong_forgot_triple(nums))
print("đúng:", Solution().unequalTriplets(nums))
assert wrong_forgot_triple(nums) != Solution().unequalTriplets(nums)

sai : 10
đúng: 0


Vì sao sai: với `[1,1,1,1,1]`, `C(5,2) * (5 - 5) = 0` nên không trừ được gì, trong khi cả `C(5,3) = 10` bộ ba đều không hợp lệ (ba số bằng nhau). Bộ ba "có đúng hai số bằng" và "cả ba bằng" là hai loại khác nhau, phải trừ cả hai. **Sửa:** trừ thêm `math.comb(c, 3)`.

### Lỗi 4 (bonus): đếm bộ ba có thứ tự

Nếu cho `i`, `j`, `k` cùng chạy `range(n)` và kiểm tra khác nhau, mỗi bộ ba bị đếm `3! = 6` lần. Hãy luôn ép `i < j < k` (hoặc chia đáp án cho 6).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm bộ (cặp, ba) mà các phần tử **đôi một khác nhau**" → gom theo giá trị bằng `Counter`, đếm theo **nhóm** thay vì theo từng phần tử.
- `n <= 100` + bộ ba → brute force `O(n^3)` vẫn qua được; hãy viết nó trước làm oracle, rồi tối ưu.
- Ba nhóm khác nhau, nhân số phần tử: gợi ý dùng đa thức đối xứng `e3` (DP `e1, e2, e3`) — kỹ thuật này mở rộng cho "bộ `k` phần tử đôi một khác nhau".
- Đếm bù: nếu "số bộ hợp lệ" khó nhưng "số bộ vi phạm" dễ phân loại → lấy tổng trừ bù.
- Sau khi sắp xếp, nhóm ở giữa có `left * c * right` — mẫu hay gặp trong bài đếm bộ ba.
- Luôn kiểm tra mẫu `[1,1,1,...]` (đáp án 0) và `n = 3`.

## 15. Biến thể của bài

1. Đếm cặp `(i, j)` với `i < j` và `nums[i] != nums[j]` (dễ hơn: `C(n,2) - Σ C(c,2)`).
2. Đếm **bộ bốn** đôi một khác nhau (đa thức đối xứng `e4`, thêm biến `e4 += e3 * c`).
3. Đếm bộ ba `(i, j, k)` với `nums[i] < nums[j] < nums[k]` (tăng ngặt, bài khác về Fenwick/DP).
4. Trả về **chính các bộ ba** (liệt kê) thay vì chỉ đếm — chỉ chạy được khi `n` nhỏ.
5. Đếm bộ ba mà **có ít nhất hai số khác nhau** (đáp án: `C(n,3) - Σ C(c,3)`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | ba vòng lặp kiểm tra đôi một khác nhau | `O(n^3)` | `O(1)` | `n` nhỏ, oracle test |
| 2. `Counter` + nhóm ở giữa | `left * c * right` trên giá trị đã sắp xếp | `O(n + k log k)` | `O(k)` | dễ giải thích |
| 3. DP `e1, e2, e3` | cập nhật tuần tự theo nhóm | `O(n)` | `O(k)` | tuyến tính, mở rộng bộ `k` |
| 4. Bù trừ | `C(n,3)` trừ các bộ có trùng | `O(n)` | `O(k)` | thích đếm bù, công thức gọn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `unequalTriplets([2, 2, 3, 3, 5])` bằng bao nhiêu?

**Bài 2 (tính tay):** với `nums = [1, 1, 2, 3, 3]`, tính bằng công thức `left * c * right` cho từng nhóm rồi cộng lại.

**Bài 3 (điền TODO):** hoàn thành `count_pairs_unequal(nums)` đếm số cặp `(i < j)` với `nums[i] != nums[j]` bằng `Counter`.

**Bài 4 (tự viết lại):** viết `unequal_quadruplets(nums)` đếm bộ **bốn** chỉ số `i < j < k < l` đôi một khác nhau (gợi ý: thêm biến `e4`).

**Bài 5 (mini challenge):** viết `list_triplets(nums)` trả về danh sách các bộ chỉ số `(i, j, k)` hợp lệ (dùng brute force) rồi kiểm tra `len(...)` bằng `Solution().unequalTriplets`.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().unequalTriplets([2, 2, 3, 3, 5]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def count_pairs_unequal(nums):
    n = len(nums)
    total = math.comb(n, 2)
    # TODO: với mỗi c trong Counter(nums).values(), trừ đi math.comb(c, 2)
    return total

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** một bộ ba hợp lệ chọn từ **ba nhóm khác nhau**; hãy gom các số bằng nhau lại (`Counter`).

**Mức 2 (vừa):** Bài 3: số cặp có thể chọn là `C(n, 2)`; số cặp **bằng nhau** trong nhóm `c` là `C(c, 2)`. Bài 4: thêm biến `e4`, cập nhật `e4 += e3 * c` **trước** `e3 += e2 * c`. Bài 5: ba vòng `for` với `i < j < k`.

**Mức 3 (gần đáp án):** Bài 1: nhóm `{2: 2, 3: 2, 5: 1}`; nhóm giữa là 3: `left = 2`, `c = 2`, `right = 1` → `2 * 2 * 1 = 4`. Bài 2: nhóm `{1:2, 2:1, 3:2}`; giữa là `2`: `2 * 1 * 2 = 4`; các nhóm kia đóng góp 0 → 4.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1: nhóm {2:2, 3:2, 5:1}; giữa là 3 -> 2 * 2 * 1 = 4
assert Solution().unequalTriplets([2, 2, 3, 3, 5]) == 4

# Bài 2: [1,1,2,3,3] -> nhóm 1: left=0 ; nhóm 2: left=2, c=1, right=2 -> 4 ; nhóm 3: right=0
assert Solution().unequalTriplets([1, 1, 2, 3, 3]) == 4

# Bài 3
def count_pairs_unequal(nums):
    n = len(nums)
    total = math.comb(n, 2)
    for c in Counter(nums).values():
        total -= math.comb(c, 2)
    return total
assert count_pairs_unequal([1, 1, 2]) == 2
assert count_pairs_unequal([5, 5, 5]) == 0
assert count_pairs_unequal([1, 2, 3, 4]) == 6

# Bài 4
def unequal_quadruplets(nums):
    e1 = e2 = e3 = e4 = 0
    for c in Counter(nums).values():
        e4 += e3 * c
        e3 += e2 * c
        e2 += e1 * c
        e1 += c
    return e4
assert unequal_quadruplets([1, 2, 3, 4]) == 1
assert unequal_quadruplets([1, 1, 2, 3, 4]) == 2
assert unequal_quadruplets([1, 1, 1, 2, 3]) == 0

# Bài 5
def list_triplets(nums):
    n = len(nums)
    return [(i, j, k) for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n)
            if len({nums[i], nums[j], nums[k]}) == 3]
assert list_triplets([4, 4, 2, 4, 3]) == [(0, 2, 4), (1, 2, 4), (2, 3, 4)]
assert len(list_triplets([1, 1, 2, 3, 3])) == Solution().unequalTriplets([1, 1, 2, 3, 3])
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm bộ ba đôi một khác nhau theo nhóm":**

```python
from collections import Counter

e1 = e2 = e3 = 0
for c in Counter(nums).values():
    e3 += e2 * c      # luôn cập nhật bậc cao trước
    e2 += e1 * c
    e1 += c
return e3
```

**Checklist trước khi submit:**

- [ ] Hiểu rằng đếm **chỉ số** (không phải giá trị phân biệt)?
- [ ] Đủ ba điều kiện khác nhau `(i,j)`, `(i,k)`, `(j,k)` (nếu dùng brute force)?
- [ ] Cập nhật `e3` trước `e2` trước `e1` (dùng giá trị cũ)?
- [ ] Nếu bù trừ: đã trừ cả "đúng hai bằng nhau" **và** "cả ba bằng nhau"?
- [ ] Đã thử `n = 3`, tất cả bằng nhau, tất cả khác nhau (`C(n,3)`)?
- [ ] Đã đối chiếu các cách với brute force bằng test ngẫu nhiên có seed?
- [ ] Chữ ký đúng `unequalTriplets(self, nums: List[int]) -> int`?